# 20 — 29042675 Semantic Recovery

## Objetivo

Determinar, antes de construir etiquetas de entrenamiento, dos aspectos:

1. Qué representan realmente las columnas asociadas a H1, H4, H5 y H6 en `IRR_Results.csv`.
2. Qué modalidad de fuente aportan los prototipos de `Prototype_Metadata.csv` y si puede considerarse una fuente web compatible con las variables X canónicas de la extensión.

Este notebook **no construye todavía el dataset maestro** ni renombra automáticamente campos `Compare` como `H*_violation`.


## 1. Configuración

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import numpy as np
import re
import zipfile
import xml.etree.ElementTree as ET

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

VT_DIR = (
    PUBLIC_RAW
    / "03_29042675"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_DIR:", PROJECT_DIR)
print("VT_DIR:", VT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning
VT_DIR: /Users/uriel/Documents/machine_learning/data/public_v2/raw/03_29042675


## 2. Cargar las tablas principales

In [2]:
metadata = pd.read_csv(
    VT_DIR
    / "Prototype_Metadata.csv"
)

irr = pd.read_csv(
    VT_DIR
    / "IRR_Results.csv"
)

print(
    "Metadata:",
    metadata.shape
)

print(
    "IRR:",
    irr.shape
)

print(
    "\nMetadata columns:"
)

print(
    metadata.columns.tolist()
)

print(
    "\nIRR columns:"
)

for i, col in enumerate(
    irr.columns
):
    print(
        i,
        repr(col)
    )


Metadata: (20, 4)
IRR: (104, 36)

Metadata columns:
['Prototype', 'Prototype Type', 'Domain', 'Scource URL']

IRR columns:
0 'Name'
1 'Prototype Link'
2 'H1 Compare'
3 'H2 Compare'
4 'H3 Compare'
5 'H4 Compare'
6 'H5 Compare'
7 'H6 Compare'
8 'H7 Compare'
9 'H8 Compare'
10 'H9 Compare'
11 'H10 Compare'
12 'Unnamed: 12'
13 'Unnamed: 13'
14 'H1. Overal Rating (4)'
15 'H1'
16 'H2. Overal Rating(4)'
17 'H2'
18 'H3. Overal Rating(4)'
19 'H3'
20 'H4 Overall Rating'
21 'H4'
22 'H5 Overall Rating'
23 'H5'
24 'H6 Overall Rating'
25 'H6'
26 'H7 Overall Rating'
27 'H7'
28 'H8 Overall Rating'
29 'H8'
30 'H9 Overall Rating'
31 'H9'
32 'H10 Overall Rating'
33 'H10'
34 'Unnamed: 0'
35 'Unnamed: 11'


## 3. Auditar la modalidad de las fuentes

Figma y Behance se consideran prototipos/presentaciones visuales, no código HTML/CSS/JS directamente equivalente a las X web de la extensión.


In [3]:
def source_domain(
    value
):

    if pd.isna(value):
        return None

    try:
        return (
            urlparse(
                str(value)
            )
            .netloc
            .lower()
            .replace(
                "www.",
                ""
            )
        )

    except Exception:
        return None


In [4]:
def classify_prototype_source(
    value
):

    domain = source_domain(
        value
    )

    if domain is None:
        return "MISSING"

    if (
        "figma.com"
        in domain
    ):
        return "FIGMA_PROTOTYPE"

    if (
        "behance.net"
        in domain
    ):
        return "BEHANCE_PRESENTATION"

    return "OTHER_SOURCE"


In [5]:
metadata_audit = (
    metadata
    .copy()
)

metadata_audit[
    "source_domain"
] = metadata_audit[
    "Scource URL"
].apply(
    source_domain
)

metadata_audit[
    "source_modality"
] = metadata_audit[
    "Scource URL"
].apply(
    classify_prototype_source
)

metadata_audit[
    "web_dom_source_candidate"
] = (
    metadata_audit[
        "source_modality"
    ]
    ==
    "OTHER_SOURCE"
)

display(
    metadata_audit
)

print(
    "\nModalidades:"
)

print(
    metadata_audit[
        "source_modality"
    ].value_counts(
        dropna=False
    )
)


,Prototype,Prototype Type,Domain,Scource URL,source_domain,source_modality,web_dom_source_candidate
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0,figma.com,FIGMA_PROTOTYPE,False
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?,figma.com,FIGMA_PROTOTYPE,False
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063,figma.com,FIGMA_PROTOTYPE,False
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,behance.net,BEHANCE_PRESENTATION,False
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441,figma.com,FIGMA_PROTOTYPE,False
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,behance.net,BEHANCE_PRESENTATION,False
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450,figma.com,FIGMA_PROTOTYPE,False
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521,figma.com,FIGMA_PROTOTYPE,False
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925,figma.com,FIGMA_PROTOTYPE,False
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,figma.com,FIGMA_PROTOTYPE,False



Modalidades:
source_modality
BEHANCE_PRESENTATION    11
FIGMA_PROTOTYPE          8
OTHER_SOURCE             1
Name: count, dtype: int64


## 4. Prototipos educativos

In [6]:
education_prototypes = (
    metadata_audit[
        metadata_audit[
            "Domain"
        ]
        .astype(str)
        .str.contains(
            "education",
            case=False,
            na=False
        )
    ]
    .copy()
)

print(
    "Prototipos educativos:",
    len(
        education_prototypes
    )
)

display(
    education_prototypes
)


Prototipos educativos: 3


,Prototype,Prototype Type,Domain,Scource URL,source_domain,source_modality,web_dom_source_candidate
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063,figma.com,FIGMA_PROTOTYPE,False
17,True Art,Low Fidelity,Education,https://www.behance.net/gallery/165695079/True-Art-Education-App-UIUX-Case-Study/modules/934460805,behance.net,BEHANCE_PRESENTATION,False
18,Quizy,Low Fidelity,Education (Quiz app),https://www.behance.net/gallery/114288043/Quizzy-Mobile-Quiz-App/modules/652681113,behance.net,BEHANCE_PRESENTATION,False


## 5. Auditar columnas asociadas a H1, H4, H5 y H6

La presencia de nombres como `H1 Compare` no implica que sean etiquetas binarias de violación. Primero se clasifica su semántica por nombre y distribución.


In [7]:
TARGETS = [
    "H1",
    "H4",
    "H5",
    "H6"
]


In [8]:
target_related_columns = []

for col in irr.columns:

    col_text = str(
        col
    )

    matched_targets = [
        target
        for target
        in TARGETS
        if re.search(
            rf"\b{target}\b",
            col_text,
            flags=re.IGNORECASE
        )
    ]

    if not matched_targets:
        continue

    for target in matched_targets:

        target_related_columns.append({
            "target":
                target,

            "column":
                col
        })

target_column_registry = (
    pd.DataFrame(
        target_related_columns
    )
)

display(
    target_column_registry
)


,target,column
0,H1,H1 Compare
1,H4,H4 Compare
2,H5,H5 Compare
3,H6,H6 Compare
4,H1,H1. Overal Rating (4)
5,H1,H1
6,H4,H4 Overall Rating
7,H4,H4
8,H5,H5 Overall Rating
9,H5,H5


In [9]:
def classify_irr_column(
    column
):

    text = str(
        column
    ).lower()

    if (
        "compare"
        in text
    ):
        return "COMPARE"

    if (
        "overall"
        in text
        or
        "overal"
        in text
    ):
        return "OVERALL_RATING"

    if (
        "rating"
        in text
    ):
        return "RATING"

    if (
        "agree"
        in text
        or
        "agreement"
        in text
    ):
        return "AGREEMENT"

    return "OTHER"


In [10]:
target_column_registry[
    "semantic_type"
] = target_column_registry[
    "column"
].apply(
    classify_irr_column
)

display(
    target_column_registry
)


,target,column,semantic_type
0,H1,H1 Compare,COMPARE
1,H4,H4 Compare,COMPARE
2,H5,H5 Compare,COMPARE
3,H6,H6 Compare,COMPARE
4,H1,H1. Overal Rating (4),OVERALL_RATING
5,H1,H1,OTHER
6,H4,H4 Overall Rating,OVERALL_RATING
7,H4,H4,OTHER
8,H5,H5 Overall Rating,OVERALL_RATING
9,H5,H5,OTHER


## 6. Distribución de columnas target-relacionadas

In [11]:
column_audit_records = []

for _, row in (
    target_column_registry
    .iterrows()
):

    target = row[
        "target"
    ]

    col = row[
        "column"
    ]

    values = pd.to_numeric(
        irr[
            col
        ],
        errors="coerce"
    )

    unique_values = (
        sorted(
            values
            .dropna()
            .unique()
            .tolist()
        )
    )

    column_audit_records.append({
        "target":
            target,

        "column":
            col,

        "semantic_type":
            row[
                "semantic_type"
            ],

        "non_missing":
            int(
                values.notna().sum()
            ),

        "missing":
            int(
                values.isna().sum()
            ),

        "unique_count":
            len(
                unique_values
            ),

        "unique_values":
            str(
                unique_values[
                    :30
                ]
            ),

        "min":
            (
                values.min()
                if values.notna().any()
                else np.nan
            ),

        "max":
            (
                values.max()
                if values.notna().any()
                else np.nan
            )
    })

irr_target_column_audit = (
    pd.DataFrame(
        column_audit_records
    )
)

display(
    irr_target_column_audit
)


,target,column,semantic_type,non_missing,missing,unique_count,unique_values,min,max
0,H1,H1 Compare,COMPARE,58,46,6,"[0.0, 1.0, 5.0, 8.0, 9.0, 10.0]",0.0,10.0
1,H4,H4 Compare,COMPARE,58,46,5,"[0.0, 1.0, 6.0, 7.0, 10.0]",0.0,10.0
2,H5,H5 Compare,COMPARE,58,46,5,"[0.0, 1.0, 7.0, 9.0, 10.0]",0.0,10.0
3,H6,H6 Compare,COMPARE,58,46,5,"[0.0, 1.0, 4.0, 6.0, 10.0]",0.0,10.0
4,H1,H1. Overal Rating (4),OVERALL_RATING,40,64,2,"[2.0, 3.0]",2.0,3.0
5,H1,H1,OTHER,0,104,0,[],NaN,NaN
6,H4,H4 Overall Rating,OVERALL_RATING,40,64,2,"[2.0, 3.0]",2.0,3.0
7,H4,H4,OTHER,0,104,0,[],NaN,NaN
8,H5,H5 Overall Rating,OVERALL_RATING,40,64,3,"[2.0, 3.0, 4.0]",2.0,4.0
9,H5,H5,OTHER,0,104,0,[],NaN,NaN


## 7. Clasificar tipos de fila del IRR

In [12]:
target_cols = (
    target_column_registry[
        "column"
    ]
    .drop_duplicates()
    .tolist()
)

def classify_irr_row(
    row
):

    has_name = (
        pd.notna(
            row.get(
                "Name"
            )
        )
        and
        str(
            row.get(
                "Name"
            )
        ).strip()
        != ""
    )

    target_values = [
        row.get(
            col
        )
        for col in target_cols
    ]

    has_target_value = any(
        pd.notna(
            value
        )
        for value in target_values
    )

    if has_name:
        return "PROTOTYPE_ROW"

    if has_target_value:
        return "SUMMARY_OR_STAT_ROW"

    return "EMPTY_OR_SEPARATOR"


In [13]:
irr_audit = (
    irr
    .copy()
)

irr_audit[
    "row_role"
] = irr_audit.apply(
    classify_irr_row,
    axis=1
)

print(
    irr_audit[
        "row_role"
    ].value_counts(
        dropna=False
    )
)

preview_cols = (
    [
        "Name",
        "Prototype Link",
        "row_role"
    ]
    +
    target_cols
)

display(
    irr_audit[
        preview_cols
    ]
)


row_role
SUMMARY_OR_STAT_ROW    58
PROTOTYPE_ROW          40
EMPTY_OR_SEPARATOR      6
Name: count, dtype: int64


,Name,Prototype Link,row_role,H1 Compare,H4 Compare,H5 Compare,H6 Compare,H1. Overal Rating (4),H1,H4 Overall Rating,H4,H5 Overall Rating,H5,H6 Overall Rating,H6
0,Food Order,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App-(Community)?node-id=0-1&p=f&t=Dx8Q70dOPPnSAAPO-0,PROTOTYPE_ROW,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gaira (Community app),https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0,PROTOTYPE_ROW,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E-Tutor,https://www.figma.com/community/file/1271304360859402063,PROTOTYPE_ROW,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tracking App,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,PROTOTYPE_ROW,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Travel,https://www.figma.com/community/file/1379689864793179441,PROTOTYPE_ROW,1.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Running,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,PROTOTYPE_ROW,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Bakery,https://www.figma.com/community/file/988854619809579450,PROTOTYPE_ROW,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Hospital 1,https://www.figma.com/community/file/1250314760992299521,PROTOTYPE_ROW,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Hospital 2,https://www.figma.com/community/file/12725823068122925,PROTOTYPE_ROW,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Housing,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,PROTOTYPE_ROW,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Número de apariciones por prototipo

In [14]:
prototype_occurrences = (
    irr_audit[
        irr_audit[
            "row_role"
        ]
        ==
        "PROTOTYPE_ROW"
    ]
    ["Name"]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "prototype_name"
    )
    .reset_index(
        name="row_count"
    )
)

display(
    prototype_occurrences
)


,prototype_name,row_count
0,Food Order,4
1,Gaira (Community app),4
2,Travel,4
3,Running,4
4,Bakery,4
5,Hospital 1,4
6,Hospital 2,4
7,Housing,4
8,E-Tutor,3
9,Tracking App,3


## 9. Cobertura de nombres entre metadata e IRR

In [15]:
def normalize_prototype_name(
    value
):

    if pd.isna(value):
        return None

    value = (
        str(value)
        .lower()
        .strip()
    )

    value = re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

    return value


In [16]:
metadata_audit[
    "prototype_norm"
] = metadata_audit[
    "Prototype"
].apply(
    normalize_prototype_name
)

irr_audit[
    "prototype_norm"
] = irr_audit[
    "Name"
].apply(
    normalize_prototype_name
)

metadata_names = set(
    metadata_audit[
        "prototype_norm"
    ]
    .dropna()
)

irr_names = set(
    irr_audit[
        "prototype_norm"
    ]
    .dropna()
)

print(
    "Metadata prototypes:",
    len(
        metadata_names
    )
)

print(
    "IRR prototype names:",
    len(
        irr_names
    )
)

print(
    "Coincidencias:",
    len(
        metadata_names
        &
        irr_names
    )
)

print(
    "\nSolo metadata:"
)

print(
    sorted(
        metadata_names
        -
        irr_names
    )
)

print(
    "\nSolo IRR:"
)

print(
    sorted(
        irr_names
        -
        metadata_names
    )
)


Metadata prototypes: 19
IRR prototype names: 11
Coincidencias: 10

Solo metadata:
['blogapp', 'fintechmobileapp', 'fooddelivery', 'healthcareapp', 'kwiktanmembership', 'onlineshoping', 'quizy', 'trueart', 'volunterypetcare']

Solo IRR:
['trackingappnotification']


## 10. Auditar archivos locales de 29042675

In [17]:
local_files = [
    path
    for path
    in VT_DIR.rglob("*")
    if path.is_file()
]

file_registry = pd.DataFrame([
    {
        "relative_path":
            str(
                path.relative_to(
                    VT_DIR
                )
            ),

        "extension":
            path.suffix.lower(),

        "size_bytes":
            path.stat().st_size
    }
    for path in local_files
])

print(
    "Archivos locales:",
    len(
        file_registry
    )
)

print(
    file_registry[
        "extension"
    ].value_counts(
        dropna=False
    )
)

display(
    file_registry
)


Archivos locales: 47
extension
.docx    43
.csv      2
.rtf      2
Name: count, dtype: int64


,relative_path,extension,size_bytes
0,Prototype_Metadata.csv,.csv,2774
1,README.rtf,.rtf,7164
2,IRR_Results.csv,.csv,90918
3,EvaluationTemplate.rtf,.rtf,172724
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,.docx,100068
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,.docx,94011
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Running.docx,.docx,132673
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Travel.docx,.docx,55216
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Gaia.docx,.docx,64913
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,.docx,91271


## 11. Extraer evidencia semántica de DOCX

In [18]:
def extract_docx_text(
    path
):

    try:

        with zipfile.ZipFile(
            path
        ) as z:

            xml = z.read(
                "word/document.xml"
            )

        root = ET.fromstring(
            xml
        )

        texts = []

        for element in root.iter():

            if (
                element.tag
                .endswith(
                    "}t"
                )
                and
                element.text
            ):
                texts.append(
                    element.text
                )

        return " ".join(
            texts
        )

    except Exception:

        return ""


In [19]:
search_terms = [
    "Visibility of system status",
    "Consistency and standards",
    "Error prevention",
    "Recognition rather than recall",
    "Overall Rating",
    "Overal Rating",
    "Compare",
    "inter-rater",
    "agreement"
]

documentation_records = []

docx_files = [
    path
    for path
    in local_files
    if path.suffix.lower()
    ==
    ".docx"
]

for path in docx_files:

    text = extract_docx_text(
        path
    )

    text_lower = (
        text.lower()
    )

    for term in search_terms:

        index = text_lower.find(
            term.lower()
        )

        if index == -1:
            continue

        start = max(
            0,
            index - 250
        )

        end = min(
            len(text),
            index + 500
        )

        documentation_records.append({
            "file":
                str(
                    path.relative_to(
                        VT_DIR
                    )
                ),

            "term":
                term,

            "context":
                text[
                    start:end
                ]
        })

documentation_evidence = (
    pd.DataFrame(
        documentation_records,
        columns=[
            "file",
            "term",
            "context"
        ]
    )
)

display(
    documentation_evidence
)


,file,term,context
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,Visibility of system status,aluation Repository Information Website URL : https://www.figma.com/community/file/12725823068122925 University Name: [University n ame] User Task : [Brief description of user task ] Heuristic Evaluation 1. The heuristic principle of 'Visibility of System Status' states that a sys...
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,Consistency and standards,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 4. The heuristic principle of Consistency and Standards Users should not hav...
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,Error prevention,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 5. The heuristic principle of Error Prevention Better than good error mess...
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,Recognition rather than recall,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 6. The heuristic principle of Recognition Rather Than Recall Minimize user...
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,Overall Rating,"running website , identify any issues related to Visibility of System Status and provide recommendations for improvement based on heuristic evaluation for the pages . W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the lofi protot..."
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,Visibility of system status,uation Repository Information Website URL : https://www.figma.com/community/file/1250314760992299521 University Name: [University n ame] User Task : [Brief description of user task ] Heuristic Evaluation 1. The heuristic principle of 'Visibility of System Status' states that a sys...
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,Consistency and standards,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 4. The heuristic principle of Consistency and Standards Users should not hav...
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,Error prevention,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 5. The heuristic principle of Error Prevention Better than good error mess...
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,Recognition rather than recall,on heuristic evaluation for the pages. W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the hifi prototype . give the answer in short and in points 6. The heuristic principle of Recognition Rather Than Recall Minimize user...
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,Overall Rating,"running website , identify any issues related to Visibility of System Status and provide recommendations for improvement based on heuristic evaluation for the pages . W ith issue give severity ratings on the scale of 0 to 4 of each issue and give overall rating of the pages of the lofi protot..."


## 12. Decisión provisional de semántica de fuente

In [20]:
source_semantics_decision = pd.DataFrame([
    {
        "dataset":
            "29042675",

        "metadata_instances":
            len(
                metadata_audit
            ),

        "educational_instances":
            len(
                education_prototypes
            ),

        "observed_source_modalities":
            " | ".join(
                sorted(
                    metadata_audit[
                        "source_modality"
                    ]
                    .dropna()
                    .unique()
                )
            ),

        "canonical_web_X_ready":
            False,

        "direct_nielsen_Y_ready":
            False,

        "current_role":
            "SEMANTIC_AUDIT",

        "reason":
            "Las fuentes observadas son prototipos visuales Figma/Behance y la semántica exacta de las columnas IRR debe confirmarse antes de construir Y."
    }
])

display(
    source_semantics_decision
)


,dataset,metadata_instances,educational_instances,observed_source_modalities,canonical_web_X_ready,direct_nielsen_Y_ready,current_role,reason
0,29042675,20,3,BEHANCE_PRESENTATION | FIGMA_PROTOTYPE | OTHER_SOURCE,False,False,SEMANTIC_AUDIT,Las fuentes observadas son prototipos visuales Figma/Behance y la semántica exacta de las columnas IRR debe confirmarse antes de construir Y.


## 13. Guardar resultados

In [21]:
metadata_audit.to_csv(
    REPORTS_DIR
    / "vt_prototype_source_audit.csv",
    index=False
)

education_prototypes.to_csv(
    REPORTS_DIR
    / "vt_education_prototypes.csv",
    index=False
)

target_column_registry.to_csv(
    REPORTS_DIR
    / "vt_irr_target_column_registry.csv",
    index=False
)

irr_target_column_audit.to_csv(
    REPORTS_DIR
    / "vt_irr_target_column_audit.csv",
    index=False
)

prototype_occurrences.to_csv(
    REPORTS_DIR
    / "vt_irr_prototype_occurrences.csv",
    index=False
)

file_registry.to_csv(
    REPORTS_DIR
    / "vt_local_file_registry.csv",
    index=False
)

documentation_evidence.to_csv(
    REPORTS_DIR
    / "vt_documentation_evidence.csv",
    index=False
)

source_semantics_decision.to_csv(
    REPORTS_DIR
    / "vt_source_semantics_decision.csv",
    index=False
)


In [22]:
expected = [
    "vt_prototype_source_audit.csv",
    "vt_education_prototypes.csv",
    "vt_irr_target_column_registry.csv",
    "vt_irr_target_column_audit.csv",
    "vt_irr_prototype_occurrences.csv",
    "vt_local_file_registry.csv",
    "vt_documentation_evidence.csv",
    "vt_source_semantics_decision.csv"
]

for filename in expected:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


vt_prototype_source_audit.csv ✅
vt_education_prototypes.csv ✅
vt_irr_target_column_registry.csv ✅
vt_irr_target_column_audit.csv ✅
vt_irr_prototype_occurrences.csv ✅
vt_local_file_registry.csv ✅
vt_documentation_evidence.csv ✅
vt_source_semantics_decision.csv ✅


## Punto de detención

No construir todavía `H1_violation`, `H4_violation`, `H5_violation` o `H6_violation`.

El siguiente paso depende de los resultados de:

- `vt_irr_target_column_audit.csv`
- `vt_irr_prototype_occurrences.csv`
- `vt_documentation_evidence.csv`
- distribución de `row_role`

Si la documentación demuestra qué campos constituyen etiquetas Nielsen directas, el siguiente notebook será la armonización de Y de 29042675.
